In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent 
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


In [3]:
duck.execute("""
CREATE OR REPLACE MACRO clean_account_name(str) AS (
    SELECT 
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    lower(
                        strip_accents(
                            replace(str, '&#38;', '')
                        )
                    ), 
                    -- 1. Suffixes juridiques et tout ce qui suit
                    '(\\b(\\s|\\))+(gmbh|mbh|g?ag|ev|kg|kgaa|se|llp|ek|ohg|ug|inc|ltd|corp|plc|gemeinnutzige)\\b).*$',
                    ''
                ),
                -- 2. Villes allemandes et variantes (nettoyées par strip_accents et lower)
                -- L'ordre privilégie les noms composés (ex: frankfurt am main) avant les noms simples
                '\\b(' ||
                -- Autour de Berlin
                'berlin|potsdam|cottbus|' ||
                -- Zone Rhénanie / Ruhr (Düsseldorf, Köln, Viersen...)
                'dusseldorf|koln|cologne|bonn|dortmund|essen|duisburg|bochum|wuppertal|monchengladbach|gelsenkirchen|aachen|krefeld|oberhausen|leverkusen|neuss|viersen|' ||
                -- Sud (Stuttgart, München...)
                'stuttgart|karlsruhe|mannheim|freiburg|heidelberg|heilbronn|muenchen|munchen|munich|augsburg|nurnberg|nuremberg|regensburg|' ||
                -- Nord (Hamburg, Kiel, Rostock...)
                'hamburg 1|hamburg|bremen|kiel|lubeck|flensburg|rostock|schwerin|' ||
                -- Centre (Frankfurt, Hannover...)
                'frankfurt am main|frankfurt|wiesbaden|mainz|darmstadt|offenbach|hannover|braunschweig|wolfsburg|kassel|' ||
                -- Divers
                'ueberregional|leipzig|dresden|chemnitz|magdeburg|halle|erfurt|' ||
                'germany' ||
                ')\\b',
                '',
                'g'
            ),
            -- 3. Nettoyage final (caractères spéciaux et espaces doubles)
            '[^a-z0-9]',
            '',
            'g'
        )
);
""")


In [3]:
duck.execute("install excel; load excel;")

In [4]:
duck.sql(
    """
    create or replace table wochenliste as 
    select * replace("ID Nummer"::int64 as "ID Nummer") 
    from read_xlsx('/Users/adrienblanquer/Downloads/2026_KW10_Wochenliste.xlsm', sheet='Kunden', header=True, range='B5:D998')
    """)

In [6]:
duck.sql(
    """
    select 
        count(*) 
    from wochenliste
    where 'Betriebsstätte' in Firmenname 

    union all

    select 
        count(*)
    from wochenliste
    where length("ID Nummer"::varchar) = 10
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          156 │
│          186 │
└──────────────┘

In [7]:
duck.sql(
    """
        select 
        *
    from wochenliste
    where length("ID Nummer"::varchar) = 10 and 'Betriebsstätte' not in Firmenname
"""
)

┌──────────────────────────────────────────────────────────────────────────────────────┬────────────┬───────────────────┐
│                                      Firmenname                                      │ ID Nummer  │     Standort      │
│                                       varchar                                        │   int64    │      varchar      │
├──────────────────────────────────────────────────────────────────────────────────────┼────────────┼───────────────────┤
│ dip Deutsche Industrie- und Parkhausbau GmbH - Standort Berlin                       │ 1040000262 │ Berlin            │
│ dip Deutsche Industrie- und Parkhausbau GmbH - Standort Hannover                     │ 1040000263 │ Hamburg           │
│ dip Deutsche Industrie- und Parkhausbau GmbH - Standort München                      │ 1040000265 │ München           │
│ dip Deutsche Industrie- und Parkhausbau GmbH - Standort Stuttgart                    │ 1040000264 │ Stuttgart         │
│ dip Deutsche Industrie

# up to date zoho accounts

In [8]:
duck.sql(
    """
    create or replace table easybill_zoho_acitve_accounts as
    with zoho_data as (
        select
            *,
            clean_account_name("Accounts Name") as clean_account_name
        from read_csv('/Users/adrienblanquer/Downloads/Accounts_2026_02_03.csv', types={'Shipping Code': 'VARCHAR'})
    ), zoho_accounts as (
        select 
            "Record ID" as id_zoho,
            "Accounts Name",
            clean_account_name("Accounts Name") as clean_account_name,
            Kundennummer,
            left(Kundennummer::varchar, 9) as id_easybill
        from zoho_data
        where Kundennummer is not null
        order by "Accounts Name"
    ), zoho_accounts_missing as (
        select
            *,
            clean_account_name(Firmenname) as clean_firmenname
        from wochenliste
        left join zoho_accounts 
            on left(wochenliste."ID Nummer"::varchar, 9) = zoho_accounts.id_easybill
        where zoho_accounts."Accounts Name" is null
    )
    select
        concat_ws('_', "ID Nummer", zoho_accounts.id_zoho) as id,
        "ID Nummer" as id_easybill,
        id_zoho,
        Firmenname,
        zoho_accounts."Accounts Name" as zoho_account_name,
        'zoho_kundennummer' as join_type,
        Standort
    from wochenliste
    join zoho_accounts
        on left("ID Nummer"::varchar, 9) = zoho_accounts.id_easybill 
    
    union all

    select
        concat_ws('_', "ID Nummer", "Record ID") as id,
        "ID Nummer" as id_easybill,
        "Record ID" as id_zoho,
        Firmenname,
        zoho_data."Accounts Name" as zoho_account_name,
        'zoho_clean_account_name' as join_type,
        Standort
    from zoho_accounts_missing
    left join zoho_data
        on zoho_accounts_missing.clean_firmenname = zoho_data.clean_account_name
    """
)

In [10]:
duck.sql(
    """
    select * from easybill_zoho_acitve_accounts
    """
)

┌────────────────────────────────────┬─────────────┬─────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────┬─────────────────────────┬────────────┐
│                 id                 │ id_easybill │         id_zoho         │                                     Firmenname                                     │                         zoho_account_name                          │        join_type        │  Standort  │
│              varchar               │    int64    │         varchar         │                                      varchar                                       │                              varchar                               │         varchar         │  varchar   │
├────────────────────────────────────┼─────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────────────────┼─────────────────────

In [11]:
duck.sql(
    """
    select 
        count(distinct id)
    from easybill_zoho_acitve_accounts
    left join pg.easybill.posten
        on left(easybill_zoho_acitve_accounts.id_easybill::varchar, 9) = posten."Kontakt: Kundennummer"
    """
)

┌────────────────────┐
│ count(DISTINCT id) │
│       int64        │
├────────────────────┤
│                997 │
└────────────────────┘

# find if there should be a medisoft account linked based on basic care invoicing or not

In [12]:
duck.sql(
    """
    select
        *
    from pg.easybill.posten
    """
)

┌───────────────────────┬────────────┬───────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────┬────────────────┬───────────────────────────┬────────────────────────────┬───────────────────────────────────┬───────────────────┬──────────────────────┬────────────────────────────┬────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────┬────────────────┬────────────────┬──────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┐
│ Kontakt: Kundennummer │ Posten: ID │ Posten: Artikelnummer │                                                                                         Posten: Artikelbeschreibung                      

# match eb x zoho match with medisoft 

In [12]:
duck.sql("from pg.easybill.posten")

┌───────────────────────┬────────────┬───────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────┬────────────────┬───────────────────────────┬────────────────────────────┬───────────────────────────────────┬───────────────────┬──────────────────────┬────────────────────────────┬────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────┬────────────────┬────────────────┬──────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┐
│ Kontakt: Kundennummer │ Posten: ID │ Posten: Artikelnummer │                                                                                         Posten: Artikelbeschreibung                      

In [13]:
duck.sql(
    """
    create or replace table easybill_zoho_acitve_accounts as 
    select
        distinct on (id)
        id,
        id_easybill,
        id_zoho,
        Firmenname,
        zoho_account_name,
        join_type,
        Standort,
        case when sum(
            case 
                when posten."Posten: Artikelnummer" ilike '%-OMW%' then 1
                else 0
            end
        ) > 0 then true else false end as should_have_medisoft_firm,
    from easybill_zoho_acitve_accounts acc
    left join pg.easybill.posten posten
        on left(acc.id_easybill::varchar, 9)  = posten."Kontakt: Kundennummer"
    group by all
    order by join_type desc
    """
)

In [33]:
duck.sql("from pg.easybill.contacts")

┌─────────────────────┬───────────────────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────────────────────┬────────────────┬──────────────────┬───────────────┬───────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬────────────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────────────┬──────────────────────┬────────────────────┬──────────────┬───────────────────────┬─────────────────────────────────────────────────┬───────────────────────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────────┬──────────────────────────────┬──────────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┬─────────────

In [ ]:
duck.sql(
    """
    with easybill_zoho_active_accounts_cleaned as (
        select
            *,
            clean_account_name(Firmenname) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
    ), medisoft_data as (
        select
            rec_id as source_id,
            rec_id as id_medisoft,
            
    )
"""
    
)

┌────────────────────────────────────┬─────────────┬─────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┬───────────────────────────┬──────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────┐
│                 id                 │ id_easybill │         id_zoho         │                                       Firmenname                                       │                         zoho_account_name                         │     join_type     │     Standort      │ should_have_medisoft_firm │                   clean_firmenname                   │                 clean_zoho_account_name                 │
│              varchar               │    int64    │         varchar         │                                        varchar                                         

In [21]:
duck.sql(
    """
    select 
        distinct on (left(id_easybill::varchar, 9))
        Firmenname,
        split(Firmenname, '//')[1] as firmenname_split,
        clean_account_name(firmenname_split) as clean_firmenname,
        clean_account_name(zoho_account_name) as clean_zoho_account_name
    from easybill_zoho_acitve_accounts
    --join pg.easybill.posten
    --    on left(easybill_zoho_acitve_accounts.id_easybill::varchar, 9) = posten."Kontakt: Kundennummer"
    """
)

┌─────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────┬─────────────────────────────────────────────┐
│                                   Firmenname                                    │                            firmenname_split                            │              clean_firmenname               │           clean_zoho_account_name           │
│                                     varchar                                     │                                varchar                                 │                   varchar                   │                   varchar                   │
├─────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ Ki

In [30]:
duck.sql(
    f"""
    with medisoft_data as (
        select 
            rec_id as id_medisoft,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as mother_entity_name, 
            name, 
            kuerzel,
            pfad,
            split(pfad, '/') as s,

            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
        where (mother_entity_name != 'Nicht Kunden' and mother_entity_name != 'Kunden')
    ), easybill_zoho_data as (
        select 
            distinct on (left(id_easybill::varchar, 9))
            *,
            split(Firmenname, '//')[2] as firmenname_split,
            clean_account_name(firmenname_split) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
    )
    select 
        id,
        id_easybill,
        id_zoho,
        id_medisoft,
        case when length(id_easybill::varchar) > 9 then true else false end as multisite,
        should_have_medisoft_firm,
        case when length(id_easybill::varchar) > 9 then split(Firmenname, '-')[1] else Firmenname end as easybill_name,
        zoho_account_name as zoho_name,
        mother_entity_name as medisoft_mother_name,
        coalesce(medisoft_data.name, medisoft_data.kuerzel) as medisoft_name,
        pfad as medisoft_path,
        s as medisoft_splited_path,
        join_type,
        jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) as sim
    from easybill_zoho_data
    left join medisoft_data
        on jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
        or jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
    where 'medicover' in lower(pfad)
    order by easybill_name
    --where should_have_medisoft_firm and id_medisoft is null
    """
)
#.to_csv('wochenliste_conso.csv')

┌────────────────────────────────────┬─────────────┬─────────────────────────┬───────────────┬───────────┬───────────────────────────┬─────────────────────────────────┬──────────────────────────────────────────────────┬──────────────────────┬──────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────┬───────────────────┬────────┐
│                 id                 │ id_easybill │         id_zoho         │  id_medisoft  │ multisite │ should_have_medisoft_firm │          easybill_name          │                    zoho_name                     │ medisoft_mother_name │                medisoft_name                 │                             medisoft_path                              │                              medisoft_splited_path                               │     join_type     │  sim   │
│              varchar               │    int64   

In [39]:
duck.sql(
    f"""
    with medisoft_data as (
        select 
            rec_id as id_medisoft,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as mother_entity_name, 
            name, 
            kuerzel,
            pfad,
            split(pfad, '/') as s,

            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
        where (mother_entity_name != 'Nicht Kunden' and mother_entity_name != 'Kunden')
    ), easybill_zoho_data as (
        select 
            distinct on (left(id_easybill::varchar, 9))
            *,
            coalesce(contacts."Kontakt: Firma", split(Firmenname, '//')[2]) as firmenname_split,
            clean_account_name(firmenname_split) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
        left join pg.easybill.contacts as contacts
            on left(easybill_zoho_acitve_accounts.id_easybill::varchar, 9) = contacts."Kontakt: Kundennummer"
    )
    select 
        id,
        id_easybill,
        id_zoho,
        id_medisoft,
        case when length(id_easybill::varchar) > 9 then true else false end as multisite,
        should_have_medisoft_firm,
        case when length(id_easybill::varchar) > 9 then split(Firmenname, '-')[1] else Firmenname end as easybill_name,
        zoho_account_name as zoho_name,
        mother_entity_name as medisoft_mother_name,
        coalesce(medisoft_data.name, medisoft_data.kuerzel) as medisoft_name,
        pfad as medisoft_path,
        s as medisoft_splited_path,
        join_type,
        jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) as sim
    from easybill_zoho_data
    left join medisoft_data
        on jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_firmenname) > 0.95
        --or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
        --or jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
    --where 'medicover' in lower(pfad)
    --where 'ejf' in lower(pfad)
    order by easybill_name
    --where should_have_medisoft_firm and id_medisoft is null
    """
).to_csv('wochenliste_conso.csv')
#.show(max_rows=1000)

In [157]:
duck.sql(
    """
        select 
            distinct on (left(id_easybill::varchar, 9))
            case when length(id_easybill::varchar) > 9 then split(Firmenname, '-')[1] else Firmenname end as easybill_name,
            *,
            split(Firmenname, '//')[1] as firmenname_split,
            clean_account_name(firmenname_split) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
        where length(id_easybill::varchar) > 9
        order by Firmenname
    """
)

┌──────────────────────────────────────────────────────────────────────────┬────────────────────────────────────┬─────────────┬─────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┬───────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────┐
│                              easybill_name                               │                 id                 │ id_easybill │         id_zoho         │                                           Firmenname                                            │                            zoho_account_name                            │     join_type     │     Standort      

In [ ]:
duck.sql(
    f"""
    with medisoft_data as (
        select 
            rec_id as id_medisoft,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as mother_entity_name, 
            name, 
            kuerzel,
            pfad,
            split(pfad, '/') as s,

            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
        where (mother_entity_name != 'Nicht Kunden' and mother_entity_name != 'Kunden')
    ), easybill_zoho_data as (
        select 
            distinct on (left(id_easybill::varchar, 9))
            *,
            split(Firmenname, '//')[2] as firmenname_split,
            clean_account_name(firmenname_split) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
    )
    select 
        id,
        id_easybill,
        id_zoho,
        id_medisoft,
        case when length(id_easybill::varchar) > 9 then true else false end as multisite,
        should_have_medisoft_firm,
        case when length(id_easybill::varchar) > 9 then split(Firmenname, '-')[1] else Firmenname end as easybill_name,
        zoho_account_name as zoho_name,
        mother_entity_name as medisoft_mother_name,
        coalesce(medisoft_data.name, medisoft_data.kuerzel) as medisoft_name,
        pfad as medisoft_path,
        s as medisoft_splited_path,
        join_type,
        jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) as sim
    from easybill_zoho_data
    left join medisoft_data
        on jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
        or jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
    --where 'medicover' in lower(pfad)
    order by easybill_name
    --where should_have_medisoft_firm and id_medisoft is null
    """
)
#.show(max_rows=1000)
#.to_csv('wochenliste_conso.csv')

┌────────────────────────────────────┬─────────────┬─────────────────────────┬──────────────────────────────────────┬───────────┬───────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────┬────────┐
│                 id                 │ id_easybill │         id_zoho         │             id_medisoft              │ multisite │ should_have_medisoft_firm │                  

In [128]:
duck.sql(
    """
    select 
        rec_id as id_medisoft,
        case 
            when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
            else clean_account_name(split(pfad, '/')[2])
        end as mother_entity_name, 
        name, 
        kuerzel,
        pfad,
        split(pfad, '/') as s,

        clean_account_name(name) as clean_name,
        clean_account_name(kuerzel) as clean_kuerzel
    from pg.medisoft.table_firmenstruktur
    where 'ejf' in lower(pfad)
    """
)

┌───────────────┬──────────────────────────────┬──────────────────────────────────┬──────────────────────────────────┬───────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│  id_medisoft  │      mother_entity_name      │               name               │             kuerzel              │                                 pfad                                  │                                         s                                          │           clean_name           │         clean_kuerzel          │
│    varchar    │           varchar            │             varchar              │             varchar              │                                varchar                                │                                     varchar[]                                      │            varchar             │        

In [58]:
duck.sql(
    f"""
    with medisoft_data as (
        select 
            rec_id as id_medisoft,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as mother_entity_name, 
            name, 
            kuerzel,
            pfad,
            split(pfad, '/') as s,

            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur
        where (mother_entity_name != 'Nicht Kunden' and mother_entity_name != 'Kunden')
    ), easybill_zoho_data as (
        select 
            distinct on (left(id_easybill::varchar, 9))
            *,
            split(Firmenname, '//')[2] as firmenname_split,
            clean_account_name(firmenname_split) as clean_firmenname,
            clean_account_name(zoho_account_name) as clean_zoho_account_name
        from easybill_zoho_acitve_accounts
    ), wochenliste_consolidated as (
        select 
            id,
            id_easybill,
            id_zoho,
            id_medisoft,
            case when length(id_easybill::varchar) > 9 then true else false end as multisite,
            should_have_medisoft_firm,
            case when length(id_easybill::varchar) > 9 then split(Firmenname, '-')[1] else Firmenname end as easybill_name,
            zoho_account_name as zoho_name,
            mother_entity_name as medisoft_mother_name,
            coalesce(medisoft_data.name, medisoft_data.kuerzel) as medisoft_name,
            pfad as medisoft_path,
            s as medisoft_splited_path,
            join_type,
            jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) as sim
        from easybill_zoho_data
        left join medisoft_data
            on jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_firmenname) > 0.95
            or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_firmenname) > 0.95
            or jaro_winkler_similarity(medisoft_data.clean_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
            or jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_zoho_data.clean_zoho_account_name) > 0.95
        --where 'medicover' in lower(pfad)
        order by easybill_name
        --where should_have_medisoft_firm and id_medisoft is null
    )
    select
        --distinct on (b.rec_id)
        count(distinct b.rec_id)
    from wochenliste_consolidated as w
    join pg.medisoft.table_beschaeftigte as b
        on w.id_medisoft = b.abetrieb_id
        or w.id_medisoft = b.ebetrieb_id
    where id_medisoft is not null
    
    union all

    select count(distinct b.rec_id)
    from pg.medisoft.table_beschaeftigte as b
    """
)
#.show(max_rows=1000)
#.to_csv('wochenliste_conso.csv')

┌──────────────────────────┐
│ count(DISTINCT b.rec_id) │
│          int64           │
├──────────────────────────┤
│                    15173 │
│                    51609 │
└──────────────────────────┘

# Extract basic care clients and invoice info

In [14]:
duck.sql(
    """
    create or replace table easybill_documents as 
        select 
            "Kontakt: Kundennummer"::varchar as id_easybill,
            "Dokument: ID",
            "Dokument: Typ",
            "Dokument: Datum",
            "Dokument: Leistungsdatum Datum",
            "Dokument: Leistungsdatum von",
            "Dokument: Leistungsdatum bis",
            "Dokument: Leistungsdatum Benutzerdefiniert",
            "Posten: Artikelnummer",
            "Posten: Artikelbeschreibung",
            "Posten: Typ",
            "Posten: Nettobetrag",
            "Posten: Bruttobetrag",
            "Posten: Anzahl"
        from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-06_03_2026-11_24_25.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
        union all
        select             
            "Kontakt: Kundennummer"::varchar as id_easybill,
            "Dokument: ID",
            "Dokument: Typ",
            "Dokument: Datum",
            "Dokument: Leistungsdatum Datum",
            "Dokument: Leistungsdatum von",
            "Dokument: Leistungsdatum bis",
            "Dokument: Leistungsdatum Benutzerdefiniert",
            "Posten: Artikelnummer",
            "Posten: Artikelbeschreibung",
            "Posten: Typ",
            "Posten: Nettobetrag",
            "Posten: Bruttobetrag",
            "Posten: Anzahl"
        from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-06_03_2026-11_24_49.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
""")

In [13]:
duck.sql(
    """
    select "Posten: Anzahl" 
    from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-06_03_2026-11_24_25.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
    """)

┌────────────────┐
│ Posten: Anzahl │
│    varchar     │
├────────────────┤
│ 1              │
│ 1              │
│ 1              │
│ 105            │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ ·              │
│ ·              │
│ ·              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 15             │
│ 1              │
│ 1              │
│ 2              │
│ 1              │
├────────────────┤
│   2352 rows    │
│   (20 shown)   │
└────────────────┘

In [6]:
duck.sql(
    """
    with wochenliste_clean as (
        select distinct on (left("ID Nummer"::varchar, 9)) left("ID Nummer"::varchar, 9) as id_easybill, *
        from wochenliste
    )
    select 
    distinct on (w.id_easybill)
    *
    from wochenliste_clean as w
    left join easybill_documents d using(id_easybill)

    """
)

┌─────────────┬────────────────────────────────────────────────────────────────────────────┬────────────┬───────────────────┬──────────────┬───────────────┬─────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────────┬────────────────────────────────────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬─────────────────────┬──────────────────────┐
│ id_easybill │                                 Firmenname                                 │ ID Nummer  │     Standort      │ Dokument: ID │ Dokument: Typ │ Dokument: Datum │ Dokument: Leistungsdatum Datum │ Dokument: Leistungsdatum von │ Dokument: Leistungsdatum bis │ Dokument: Leistungsdatum Benutzerdefiniert │ Posten: Artikelnummer │                                                                             

In [24]:
duck.sql(
    """
    with wochenliste_clean as (
        select left("ID Nummer"::varchar, 9) as id_easybill, *
        from wochenliste
    )
    select distinct on (id_easybill, "Posten: Artikelnummer")
        *,
        "Posten: Anzahl" == '1' as is_pauschal
        --'pauschal' in "Posten: Artikelbeschreibung" as is_pauschal
    from wochenliste_clean as w
    left join easybill_documents d using(id_easybill)
    where "Posten: Artikelnummer" in ('AM-R', 'AS-R')
        --and id_easybill = '130002311'
    order by id_easybill, "Dokument: Datum" desc
""").to_csv('wochenliste_pauschal.csv')

In [25]:
duck.sql(
    """
    select
        --distinct on (id_easybill)
        count(distinct id_easybill) filter (where is_pauschal is true) as nb_pauschal_clients,
        count(distinct id_easybill) filter (where is_pauschal is false) as nb_non_pauschal_clients
    from read_csv('wochenliste_pauschal.csv')
    """
)

┌─────────────────────┬─────────────────────────┐
│ nb_pauschal_clients │ nb_non_pauschal_clients │
│        int64        │          int64          │
├─────────────────────┼─────────────────────────┤
│                 348 │                     475 │
└─────────────────────┴─────────────────────────┘

In [26]:
duck.sql(
    """
    with wochenliste_pauschal as (
        select 
            id_easybill,
            round(sum(replace(left("Posten: Nettobetrag", -4), ',', '.')::float), 2) as net_price,
        any_value(is_pauschal) as is_pauschal
        from read_csv('wochenliste_pauschal.csv')
        where is_pauschal is true
        group by id_easybill
    )
    select
        net_price,
        count(distinct id_easybill) as nb_clients
    from wochenliste_pauschal
    group by net_price
    order by nb_clients desc
    """
).to_csv('wochenliste_pauschal_prices.csv')

In [27]:
duck.sql(
    """
    with wochenliste_non_pauschal as (
        select 
            id_easybill,
            round(sum(replace(left("Posten: Nettobetrag", -4), ',', '.')::float), 2) as net_price,
        any_value(is_pauschal) as is_pauschal
        from read_csv('wochenliste_pauschal.csv')
        where is_pauschal is false
        group by id_easybill
    )
    select
        net_price,
        count(distinct id_easybill) as nb_clients
    from wochenliste_non_pauschal
    group by net_price
    order by nb_clients desc
    """
).to_csv('wochenliste_non_pauschal_prices.csv')

In [120]:
duck.sql(
    """
        with wochenliste_non_pauschal as (
        select 
            id_easybill,
            round(sum(replace(left("Posten: Nettobetrag", -4), ',', '.')::float), 2) as net_price,
        any_value(is_pauschal) as is_pauschal
        from read_csv('wochenliste_pauschal.csv')
        where is_pauschal is false
        group by id_easybill
    )select * from wochenliste_non_pauschal
where net_price = 2000
    """
)

┌─────────────┬───────────┬─────────────┐
│ id_easybill │ net_price │ is_pauschal │
│    int64    │  double   │   boolean   │
├─────────────┼───────────┼─────────────┤
│   111001023 │    2000.0 │ false       │
│   116010003 │    2000.0 │ false       │
│   130000529 │    2000.0 │ false       │
│   130001081 │    2000.0 │ false       │
│   130001259 │    2000.0 │ false       │
│   130001568 │    2000.0 │ false       │
│   130000041 │    2000.0 │ false       │
│   130000543 │    2000.0 │ false       │
│   130001151 │    2000.0 │ false       │
│   130001193 │    2000.0 │ false       │
│       ·     │       ·   │   ·         │
│       ·     │       ·   │   ·         │
│       ·     │       ·   │   ·         │
│   130000854 │    2000.0 │ false       │
│   130000901 │    2000.0 │ false       │
│   130001360 │    2000.0 │ false       │
│   130002140 │    2000.0 │ false       │
│   101000009 │    2000.0 │ false       │
│   108040008 │    2000.0 │ false       │
│   119020075 │    2000.0 │ false 

In [85]:
duck.sql(
    """
    select * from wochenliste where "ID Nummer" = '104000051'
    """
)

┌────────────────────────────────────────────────────────────────────────────┬───────────┬──────────┐
│                                 Firmenname                                 │ ID Nummer │ Standort │
│                                  varchar                                   │   int64   │ varchar  │
├────────────────────────────────────────────────────────────────────────────┼───────────┼──────────┤
│ Digital Ocean Lab Fraunhofer Institut fpr Graphische Datenverarbeitung IGD │ 104000051 │ Rostock  │
└────────────────────────────────────────────────────────────────────────────┴───────────┴──────────┘

In [82]:
duck.sql(
    """
    select *
    from easybill_documents
    where id_easybill= '104000051'
    """
)

┌─────────────┬──────────────┬───────────────┬─────────────────┬────────────────────────────────┬──────────────────────────────┬──────────────────────────────┬────────────────────────────────────────────┬───────────────────────┬──────────────────────────────────────────────┬────────────────┬─────────────────────┬──────────────────────┐
│ id_easybill │ Dokument: ID │ Dokument: Typ │ Dokument: Datum │ Dokument: Leistungsdatum Datum │ Dokument: Leistungsdatum von │ Dokument: Leistungsdatum bis │ Dokument: Leistungsdatum Benutzerdefiniert │ Posten: Artikelnummer │         Posten: Artikelbeschreibung          │  Posten: Typ   │ Posten: Nettobetrag │ Posten: Bruttobetrag │
│   varchar   │    int64     │    varchar    │      date       │              date              │             date             │             date             │                  varchar                   │        varchar        │                   varchar                    │    varchar     │       varchar       │       var

In [35]:
duck.sql("""
 select columns('Posten*') from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-06_03_2026-11_24_25.csv')
""")

┌────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────┬────────────────┬───────────────────────────┬────────────────────────────┬───────────────────────────────────┬───────────────────┬──────────────────────┬────────────────────────────┬────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────┬────────────────┬────────────────┬──────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┐
│ Posten: ID │ Posten: Artikelnummer │                                                        Posten: Artikelbeschreibung                                                        │ Posten: Position │  Posten: Typ   │ Posten: Einzelpreis Netto │ Posten: Einzelpreis Brutto │ Posten: Einkaufspr